In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, DoubleType, IntegerType

# 1. PATH CONFIGURATION
BRONZE_POS_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/pos_transactions_stream/"
SILVER_POS_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/silver/pos_transactions_stream/"
CHECKPOINT_SILVER_POS = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/pos_transactions_stream/"

# 2. READ STREAM FROM BRONZE
# Reading the Delta table we just created in the Bronze layer
df_bronze_pos = spark.readStream.format("delta").load(BRONZE_POS_PATH)

# 3. TRANSFORMATIONS (SILVER LAYER TASKS)
df_silver_pos = (
    df_bronze_pos
    # --- TASK: DATA TYPE ENFORCEMENT & SCHEMA STANDARDIZATION ---
    .withColumn("event_timestamp", F.col("timestamp").cast(TimestampType()))
    .withColumn("quantity",        F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price",      F.col("unit_price").cast(DoubleType()))
    .withColumn("total_amount",    F.col("total_amount").cast(DoubleType()))
    

    # --- TASK: DEDUPLICATION ---
    .dropDuplicates(["transaction_id"])
    
    # --- TASK: BUSINESS RULE VALIDATION ---
    .filter((F.col("total_amount") > 0) & (F.col("quantity") > 0))
        
    # Add Processing Metadata
    .withColumn("_silver_processing_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_source")) # Carrying over source info
)

# 4. WRITE STREAM TO SILVER FOLDER
# Using 'append' mode to maintain an audit-friendly transaction history
query_silver_pos = (
    df_silver_pos.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_SILVER_POS)
        .trigger(availableNow=True) 
        .start(SILVER_POS_PATH)
)

query_silver_pos.awaitTermination()